## 4. Hyperparameter Tuning

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from src.models import train_pipline,auto_tune
from src.utils import show_confusion_matrix,plot_roc_curve

from pytabkit import RealMLP_TD_Classifier
from pytabkit import TabM_D_Classifier
from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
raw_path = os.path.dirname(os.path.abspath('.'))

df_FE_train_path = os.path.join(raw_path, "df_FE_train.csv")
df_FE_test_path = os.path.join(raw_path, "df_FE_test.csv")

df_FE = pd.read_csv(df_FE_train_path)
df_FE_test= pd.read_csv(df_FE_test_path)

X = df_FE.drop(['Stress_Level'], axis=1)
y = df_FE['Stress_Level']

X

In [ ]:
y

In [ ]:
target_names = ['0', '1']

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
y_resampled

In [ ]:
y_resampled.value_counts()

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
LGBM_params = {
    "n_estimators":("int", 800,2000),
    "max_depth": ("int", 5,9),
    "learning_rate": ("float", 0.02, 0.15,True),
    "subsample": ("float", 0.7, 1.0, False),
    "colsample_bytree":("float", 0.6, 1.0, False),
    "num_leaves": ("int", 50, 200),
    "min_child_samples": ("int", 10, 50),
    "reg_alpha": ("float", 1e-4, 1.0, True),
    "reg_lambda": ("float", 1e-4, 1.0, True),
    'verbose':-1,
}

XGB_params = {
    "n_estimators": ("int",800,2000),
    "max_depth": ("int", 4,8),
    "learning_rate": ("float", 0.02,0.15,True),
    "subsample": ("float", 0.7, 1.0, False),
    "colsample_bytree": ("float", 0.6, 1.0, False),
    "min_child_weight": ("int", 1, 10),
    "reg_alpha": ("float", 1e-4, 1.0, True),
    "reg_lambda":("float", 1e-4, 1.0, True),
    "gamma":  ("float", 0, 0.5, False),
    'verbosity':0,
}

In [ ]:
# LGBM_study = auto_tune(X_resampled, y_resampled, model_class=LGBMClassifier, tuning_params=LGBM_params, cv=cv, n_trials=30)
# print(LGBM_study.best_params)
# print(LGBM_study.best_value)

In [ ]:
# XGB_study = auto_tune(X_resampled,y_resampled,XGBClassifier,XGB_params,cv=cv,n_trials=15)
# print(XGB_study.best_params)
# print(XGB_study.best_value)

In [ ]:
LGBM_BEST_PARAMS = {'n_estimators': 1036,
                    'max_depth': 7,
                    'learning_rate': 0.020935396373398023,
                    'subsample': 0.992166101648227,
                    'colsample_bytree': 0.7494380846087404,
                    'num_leaves': 67,
                    'min_child_samples': 45,
                    'reg_alpha': 0.0006650797129419691,
                    'reg_lambda': 0.20479605091612718}

XGB_BEST_PARAMS = {'n_estimators': 1007,
                   'max_depth': 7,
                   'learning_rate': 0.020066611690613374,
                   'subsample': 0.7013046414240078,
                   'colsample_bytree': 0.7447371598997203,
                   'min_child_weight': 9,
                   'reg_alpha': 0.0015993208674721017,
                   'reg_lambda': 0.0060640794884617094,
                   'gamma': 0.09967898557240965}

In [ ]:
RMLP_model, RLMP_y_preds, RLMP_y_probas = train_pipline(X_resampled, y_resampled, RealMLP_TD_Classifier, cv=cv)
show_confusion_matrix(y_resampled, RLMP_y_preds, target_names)
plot_roc_curve(y_resampled, RLMP_y_probas, target_names)
print(classification_report(y_resampled, RLMP_y_preds, target_names=target_names))

In [ ]:
TM_model , TM_y_preds, TM_y_probas= train_pipline(X_resampled,y_resampled,TabM_D_Classifier,cv=cv)
show_confusion_matrix(y_resampled,TM_y_preds,target_names)
plot_roc_curve(y_resampled,TM_y_probas,target_names)
print(classification_report(y_resampled,TM_y_preds,target_names=target_names))

In [ ]:
ETC_model , ETC_y_preds, ETC_y_probas= train_pipline(X_resampled,y_resampled,ExtraTreesClassifier,cv=cv)
show_confusion_matrix(y_resampled,ETC_y_preds,target_names)
plot_roc_curve(y_resampled, ETC_y_probas, target_names)
print(classification_report(y_resampled,ETC_y_preds,target_names=target_names))

In [ ]:
LGBM_model , LGBM_y_preds, LGBM_y_probas= train_pipline(X_resampled,y_resampled,LGBMClassifier,LGBM_BEST_PARAMS,cv)
show_confusion_matrix(y_resampled,LGBM_y_preds,target_names)
plot_roc_curve(y_resampled,LGBM_y_probas,target_names)
print(classification_report(y_resampled,LGBM_y_preds,target_names=target_names))

In [ ]:
XGB_model , XGB_y_preds, XGB_y_probas= train_pipline(X_resampled,y_resampled,XGBClassifier,XGB_BEST_PARAMS,cv)
show_confusion_matrix(y_resampled,XGB_y_preds,target_names)
plot_roc_curve(y_resampled,XGB_y_probas,target_names)
print(classification_report(y_resampled,XGB_y_preds,target_names=target_names))

In [ ]:
save_probas = '..\probas'
if not os.path.exists(save_probas):
    os.mkdir(save_probas)

np.save(os.path.join(save_probas, 'oof_RMLP_y_probas.npy'), RLMP_y_probas)
np.save(os.path.join(save_probas, 'oof_TM_y_probas.npy'), TM_y_probas)
np.save(os.path.join(save_probas, 'oof_ETC_y_probas.npy'), ETC_y_probas)
np.save(os.path.join(save_probas, 'oof_LGBM_y_probas.npy'), LGBM_y_probas)
np.save(os.path.join(save_probas, 'oof_XGB_y_probas.npy'), XGB_y_probas)

In [ ]:
cols_order = X_resampled.columns.tolist()

In [ ]:
RMLP_test_probas = train_pipline(X_resampled, y_resampled, RealMLP_TD_Classifier, X_test=df_FE_test[cols_order])
np.save(os.path.join(save_probas, 'RMLP_test_probas.npy'), RMLP_test_probas)

In [ ]:
TM_test_probas = train_pipline(X_resampled, y_resampled, TabM_D_Classifier, X_test=df_FE_test[cols_order])
np.save(os.path.join(save_probas, 'TM_test_probas.npy'), TM_test_probas)

In [ ]:
ETC_test_probas = train_pipline(X_resampled,y_resampled,ExtraTreesClassifier,X_test=df_FE_test[cols_order])
np.save(os.path.join(save_probas, 'ETC_test_probas.npy'), ETC_test_probas)

In [ ]:
LGBM_test_probas = train_pipline(X_resampled,y_resampled,LGBMClassifier,LGBM_BEST_PARAMS,X_test=df_FE_test[cols_order])
np.save(os.path.join(save_probas, 'LGBM_test_probas.npy'), LGBM_test_probas)

In [ ]:
XGB_test_probas = train_pipline(X_resampled,y_resampled,XGBClassifier,XGB_BEST_PARAMS,X_test=df_FE_test[cols_order])
np.save(os.path.join(save_probas, 'XGB_test_probas.npy'), XGB_test_probas)